# Conversion and Inference of unknown GSDR
General pipeline: Load testset -> Inference

Conversion to APEBench format:
1) 'uv' of unknown GSDR maps to 'sims' of APEBench . There is a need to upsample.
2) Add the corresponding attributes of the unknown GSDR to 'sims'. NOTE: This step is required, since metadata would be extracted from the attrs, even though the model does not really do anything with the parameters.
3) Ensure that the dt of both the unknown GSDR and APEBench GSDR are the same.
4) Loading and inference was done using gs_theta's pipeline


## Note:
In this notebook, I have also experimented by changing the feed rate/kill rate to differing values but the plots essentially do not change.

[Paper for unknown GSDR](https://arxiv.org/pdf/2106.04781)


# Conversion of unknown GSDR -> APEBench GSDR format

In [ ]:
import scipy.io
import numpy as np

file_path = '/Volumes/T7/New_Data/2d_gs_rd/data/2DRD_2x3001x100x100_[dt=05].mat'

data = scipy.io.loadmat(file_path)

print(data.keys())

uv_data = data['uv']
print(uv_data.shape)
print(uv_data.dtype)

num_channels = uv_data.shape[0]

for c in range(num_channels):
    channel_data = uv_data[c]  # shape: (3001, 100, 100)

    print(f"Channel {c}:")
    print("  min:", np.min(channel_data))
    print("  max:", np.max(channel_data))

# 2 channels
# 3001 timesteps
# 100 by 100

# Checking the data for the original APEBench GSDR

In [ ]:
import h5py
import numpy as np

file = '/pde-transformer/datasets/10sims_30t/gs_theta.hdf5'

with h5py.File(file, 'r') as f:
    print(f.keys())
    print(f['norm_const_max'])  # [0.04,0.06]
    print(f['norm_const_mean']) # [0.04,0.06]
    print(f['norm_const_min']) # [0.04,0.06]
    print(f['norm_const_std']) # [0,0] floating point

    print()
    print(f['norm_fields_sca_max']) # [[[1.00000179]], [[0.41890338]]]
    print(f['norm_fields_sca_mean']) # [[[0.69602691]], [[0.12202653]]]
    print(f['norm_fields_sca_min']) # [[[ 2.51750410e-01]], [[-4.32133675e-07]]]
    print(f['norm_fields_sca_std']) # [[[0.25863836]], [[0.12579946]]]

    print()
    print(f['norm_fields_std']) # [[[0.25863832]], [[0.12579946]]]

    print()
    print(f['sims'])
    print(f['sims'].keys())
    print(f['sims']['sim0'])

    print()
    for attrsKeys in f['sims'].attrs.keys():
        print(f'{attrsKeys}:',f['sims'].attrs[attrsKeys])

    print()
    print(f['sims']['sim0'].attrs['Feed Rate'])
    print(f['sims']['sim2'].attrs['Feed Rate'])
    print(f['sims']['sim7'].attrs['Feed Rate'])

    print()
    ds = f['sims']['sim0']
    num_channels = 2

    for i in range(num_channels):
        channel_data = ds[..., i]
        c_min = np.min(channel_data)
        c_max = np.max(channel_data)

        print(f"Channel {i}:")
        print(f"  Min: {c_min}")
        print(f"  Max: {c_max}")


    print(f['sims']['sim0'][0])
    print()
    print(f['sims']['sim1'][0])

    print(f['norm_const_std'][:])


# Conversion of unknown GSDR

In [ ]:
import h5py
import numpy as np
import scipy.io
from scipy.ndimage import zoom


file_path = '/Volumes/T7/New_Data/2d_gs_rd/data/2DRD_2x3001x100x100_[dt=05].mat'
data = scipy.io.loadmat(file_path)
uv_data = data['uv']  # (2,3001,100,100)

# (2,3001,100,100)-> (8,2,100,100)
step = 400
uv_data = uv_data[:, ::step, :, :]
uv_reshaped = np.transpose(uv_data, (1, 0, 2, 3))

#New Domain extent
L = 2.5
res = 256
coords = np.linspace(0, L, res, dtype=np.float32)

#Res 100x100 -> 256x256
zoom_factors = (1, 1, 256/100, 256/100)
uv_resized = zoom(uv_reshaped, zoom_factors, order=3).astype(np.float32)

print('New shape:', uv_resized.shape)

# Stats calculation
norm_max = np.max(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)
norm_mean = np.mean(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)
norm_min = np.min(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)
norm_std = np.std(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)

#Feed rate/kill rate
const_vals = np.array([0.04, 0.06], dtype=np.float64)

#HDF5 creation
output_path = '/Volumes/T7/New_Data/2d_gs_rd/data/gs_theta.hdf5'
with h5py.File(output_path, 'w') as f:

    # Stats
    f.create_dataset('norm_const_max', data=const_vals)
    f.create_dataset('norm_const_mean', data=const_vals)
    f.create_dataset('norm_const_min', data=const_vals)
    f.create_dataset('norm_const_std', data=np.array([0.0, 0.0], dtype=np.float64))

    f.create_dataset('norm_fields_sca_max', data=norm_max.astype(np.float64))
    f.create_dataset('norm_fields_sca_mean', data=norm_mean.astype(np.float64))
    f.create_dataset('norm_fields_sca_min', data=norm_min.astype(np.float64))
    f.create_dataset('norm_fields_sca_std', data=norm_std.astype(np.float64))
    f.create_dataset('norm_fields_std', data=norm_std.astype(np.float32))

    # Data
    sims_group = f.create_group('sims')
    sim0 = sims_group.create_dataset('sim0', data=uv_resized)

    sim0.attrs['Feed Rate'] = 0.04
    sim0.attrs['Kill Rate'] = 0.06

    # Metadata
    attrs = {
        'PDE': 'Gray Scott',
        'Dimension': 2,
        'Fields Scheme': 'cb',
        'Fields': ['Concentration A', 'Concentration B'],
        'Domain Extent': 2.5,
        'Resolution': 256,
        'Time Steps': 8,
        'Dt': 200.0,
        'Sub Steps': 0,
        'Constants': ['Feed Rate', 'Kill Rate'],
        'Initial Types': ['Gaussian Blobs'],
        'Diffusivity A': 2e-05,
        'Diffusivity B': 5e-06,
        'Dynamics Types (Feed Rate, Kill Rate)': [[0.04, 0.06]],
        'Boundary Conditions Order': ['x negative', 'x positive', 'y negative', 'y positive'],
        'Boundary Conditions': ['periodic', 'periodic', 'periodic', 'periodic']
    }

    for key, val in attrs.items():
        sims_group.attrs[key] = val

print('Done')
import matplotlib.pyplot as plt
# Compare a slice of original vs resized
plt.subplot(1,2,1); plt.imshow(uv_reshaped[0,0]); plt.title("Original (100x100)")
plt.subplot(1,2,2); plt.imshow(uv_resized[0,0]); plt.title("Zoomed (256x256)")
plt.show()

# Sanity check

In [ ]:
import h5py

file = '/Volumes/T7/New_Data/2d_gs_rd/data/gs_theta.hdf5'

with h5py.File(file, 'r') as f:
    print(f.keys())
    print(f['norm_const_max'])  # [0.04,0.06]
    print(f['norm_const_mean']) # [0.04,0.06]
    print(f['norm_const_min']) # [0.04,0.06]
    print(f['norm_const_std']) # [0,0] floating point

    print()
    print(f['norm_fields_sca_max']) # [[[1.00000179]], [[0.41890338]]]
    print(f['norm_fields_sca_mean']) # [[[0.69602691]], [[0.12202653]]]
    print(f['norm_fields_sca_min']) # [[[ 2.51750410e-01]], [[-4.32133675e-07]]]
    print(f['norm_fields_sca_std']) # [[[0.25863836]], [[0.12579946]]]

    print()
    print(f['norm_fields_std']) # [[[0.25863832]], [[0.12579946]]]

    print()
    print(f['sims'])
    print(f['sims'].keys())
    print(f['sims']['sim0'])

    print()
    for attrsKeys in f['sims'].attrs.keys():
        print(f'{attrsKeys}:',f['sims'].attrs[attrsKeys])

    print()
    print(f['sims']['sim0'].attrs.keys())

    print(f['norm_const_std'][:])


# Data loading of converted unknown GSDR

In [ ]:
from pdetransformer.core.separate_channels import PDETransformer, Supervised
import torch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Load pre-trained model
model = PDETransformer.from_pretrained('thuerey-group/pde-transformer', subfolder='sc-s').to(device)
strategy = Supervised(model, timesteps=2)

# Prediction for converted unknown GSDR

In [ ]:
from pdetransformer.data import MultiDataModule

dataset = 'gs_theta'
dataset_name = 'gs_theta'

params_data = {
    'path_index':
        {'2D_APE_xxl': '/Volumes/T7/New_Data/2d_gs_rd/data'},
    'dataset_names': [ dataset ],
    'dataset_type': '2D_APE_xxl',
    'unrolling_steps': 1,
    'test_unrolling_steps': 7,
    'batch_size': 1,
    'num_workers': 1,
    'cache_strategy': 'testOnly',
    'different_resolution_strategy': 'none',
    'normalize_data': 'mean-std',
    'normalize_const': 'None',
    'downsample_factor': 1,
    'max_channels': 2,
}

data_module = MultiDataModule(**params_data)
data_module.setup(stage='test')
test_loader = data_module.test_dataloader()

In [ ]:
data = next(iter(test_loader))

prediction, reference = strategy.predict(data, device=device, num_frames=8)
print('Done')

# Plotting pred and reference for converted unknown GSDR data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_theme(style="white")
cmap = sns.color_palette("twilight", as_cmap=True)

time_steps = np.arange(8)

pred_A = prediction[0, time_steps, 0]
pred_B = prediction[0, time_steps, 1]
ref_A  = reference[0, time_steps, 0]
ref_B  = reference[0, time_steps, 1]

fig, axes = plt.subplots(4, 8, figsize=(18, 8), dpi=200)

for j in range(8):
    # Row 0: Prediction A
    axes[0, j].imshow(pred_A[j], cmap=cmap)
    axes[0, j].set_title(f't={j}')

    # Row 1: Reference A
    axes[1, j].imshow(ref_A[j], cmap=cmap)

    # Row 2: Prediction B
    axes[2, j].imshow(pred_B[j], cmap=cmap)

    # Row 3: Reference B
    axes[3, j].imshow(ref_B[j], cmap=cmap)

    for i in range(4):
        axes[i, j].axis('off')

labels = ['Pred A', 'Ref A', 'Pred B', 'Ref B']

for i, label in enumerate(labels):
    ax = axes[i, 0]
    ax.axis('on')
    ax.set_ylabel(label, fontsize=14, fontweight='bold', labelpad=25)

    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

plt.tight_layout()
plt.show()

# Animations for chemical conc A(channel 0), B(channel 1)
To further illustrate the changes in prediction as time goes on.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import matplotlib as mpl
import seaborn as sns
import numpy as np

sns.set_theme(style="white")
cmap = sns.color_palette("twilight", as_cmap=True)
mpl.rcParams['animation.embed_limit'] = 100.0

pred_plot = prediction[0, :, 0, :, :]
ref_plot = reference[0, :, 0, :, :]

vmin = min(pred_plot.min(), ref_plot.min())
vmax = max(pred_plot.max(), ref_plot.max())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
im1 = ax1.imshow(pred_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)
im2 = ax2.imshow(ref_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)

ax1.set_title("Prediction (T=0)")
ax2.set_title("Reference (T=0)")
ax1.axis('off')
ax2.axis('off')

def update(i):
    im1.set_data(pred_plot[i])
    im2.set_data(ref_plot[i])

    ax1.set_title(f"Prediction (T={i})")
    ax2.set_title(f"Reference (T={i})")

    return [im1, im2]

ani = animation.FuncAnimation(
    fig,
    update,
    frames=8,
    interval=100, 
    blit=False
)

plt.close() 

display(HTML(ani.to_jshtml()))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import matplotlib as mpl
import seaborn as sns
import numpy as np

sns.set_theme(style="white")
cmap = sns.color_palette("twilight", as_cmap=True)
mpl.rcParams['animation.embed_limit'] = 100.0

pred_plot = prediction[0, :, 1, :, :]
ref_plot = reference[0, :, 1, :, :]

vmin = min(pred_plot.min(), ref_plot.min())
vmax = max(pred_plot.max(), ref_plot.max())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
im1 = ax1.imshow(pred_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)
im2 = ax2.imshow(ref_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)

ax1.set_title("Prediction (T=0)")
ax2.set_title("Reference (T=0)")
ax1.axis('off')
ax2.axis('off')

def update(i):
    im1.set_data(pred_plot[i])
    im2.set_data(ref_plot[i])

    ax1.set_title(f"Prediction (T={i})")
    ax2.set_title(f"Reference (T={i})")

    return [im1, im2]

ani = animation.FuncAnimation(
    fig,
    update,
    frames=8,
    interval=100, 
    blit=False
)

plt.close() 

display(HTML(ani.to_jshtml()))

# NRMSE check using PDEtransformer's own script

In [ ]:
from pdetransformer.metric.mse import SimulationMSE
import torch

nrmse_metric = SimulationMSE(
    frequency=1,
    num_tasks=1,
    max_length=3001,
    normalized=True,
    dataset_names=['gs_theta']
)

pred_tensor = torch.from_numpy(prediction)
ref_tensor = torch.from_numpy(reference)

class_labels = torch.tensor([0])
nrmse_metric.update(pred_tensor, ref_tensor, class_labels)

nrmse_results = nrmse_metric.compute()

print("Times tracked by metric:", nrmse_metric.times)

for i in range(len(nrmse_results[0])):
    print(f"Index {i}, timestep {nrmse_metric.times[i]}: {nrmse_results[0,i].item():.6f}")

times = nrmse_metric.times

step1_index = times.index(1)
step3000_index = times.index(3000)

nrmse_step_1 = nrmse_results[0, step1_index].item()
nrmse_step_3000 = nrmse_results[0, step3000_index].item()

print(f"NRMSE after 1 step:  {nrmse_step_1:.6f}")
print(f"NRMSE after 3000 steps: {nrmse_step_3000:.6f}")

# Change in Diffusivity B
5e-06 -> 5e-04
To check if there is a difference in predictions

In [ ]:
import h5py
import numpy as np
import scipy.io
from scipy.ndimage import zoom


file_path = '/Volumes/T7/New_Data/2d_gs_rd/data/2DRD_2x3001x100x100_[dt=05].mat'
data = scipy.io.loadmat(file_path)
uv_data = data['uv']  # (2,3001,100,100)

# (2,3001,100,100)-> (8,2,100,100)
step = 400
uv_data = uv_data[:, ::step, :, :]
uv_reshaped = np.transpose(uv_data, (1, 0, 2, 3))

#New Domain extent
L = 2.5
res = 256
coords = np.linspace(0, L, res, dtype=np.float32)

#Res 100x100 -> 256x256
zoom_factors = (1, 1, 256/100, 256/100)
uv_resized = zoom(uv_reshaped, zoom_factors, order=3).astype(np.float32)

print('New shape:', uv_resized.shape)

# Stats calculation
norm_max = np.max(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)
norm_mean = np.mean(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)
norm_min = np.min(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)
norm_std = np.std(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)

#Feed rate/kill rate: 
const_vals = np.array([0.04, 0.06], dtype=np.float64)

#HDF5 creation
output_path = '/Volumes/T7/New_Data/2d_gs_rd/data/gs_theta.hdf5'
with h5py.File(output_path, 'w') as f:

    # Stats
    f.create_dataset('norm_const_max', data=const_vals)
    f.create_dataset('norm_const_mean', data=const_vals)
    f.create_dataset('norm_const_min', data=const_vals)
    f.create_dataset('norm_const_std', data=np.array([0.0, 0.0], dtype=np.float64))

    f.create_dataset('norm_fields_sca_max', data=norm_max.astype(np.float64))
    f.create_dataset('norm_fields_sca_mean', data=norm_mean.astype(np.float64))
    f.create_dataset('norm_fields_sca_min', data=norm_min.astype(np.float64))
    f.create_dataset('norm_fields_sca_std', data=norm_std.astype(np.float64))
    f.create_dataset('norm_fields_std', data=norm_std.astype(np.float32))

    # Data
    sims_group = f.create_group('sims')
    sim0 = sims_group.create_dataset('sim0', data=uv_resized)

    sim0.attrs['Feed Rate'] = 0.04
    sim0.attrs['Kill Rate'] = 0.06

    # Metadata
    attrs = {
        'PDE': 'Gray Scott',
        'Dimension': 2,
        'Fields Scheme': 'cb',
        'Fields': ['Concentration A', 'Concentration B'],
        'Domain Extent': 2.5,
        'Resolution': 256,
        'Time Steps': 8,
        'Dt': 200.0,
        'Sub Steps': 0,
        'Constants': ['Feed Rate', 'Kill Rate'],
        'Initial Types': ['Gaussian Blobs'],
        'Diffusivity A': 2e-05,
        'Diffusivity B': 5e-04, #change by a larger magnitude to check
        'Dynamics Types (Feed Rate, Kill Rate)': [[0.04, 0.06]], #change to check prediction
        'Boundary Conditions Order': ['x negative', 'x positive', 'y negative', 'y positive'],
        'Boundary Conditions': ['periodic', 'periodic', 'periodic', 'periodic']
    }

    for key, val in attrs.items():
        sims_group.attrs[key] = val

print('Done')
import matplotlib.pyplot as plt
# Compare a slice of original vs resized
plt.subplot(1,2,1); plt.imshow(uv_reshaped[0,0]); plt.title("Original (100x100)")
plt.subplot(1,2,2); plt.imshow(uv_resized[0,0]); plt.title("Zoomed (256x256)")
plt.show()

In [ ]:
from pdetransformer.data import MultiDataModule

dataset = 'gs_theta'
dataset_name = 'gs_theta'

params_data = {
    'path_index':
        {'2D_APE_xxl': '/Volumes/T7/New_Data/2d_gs_rd/data'},
    'dataset_names': [ dataset ],
    'dataset_type': '2D_APE_xxl',
    'unrolling_steps': 1,
    'test_unrolling_steps': 7,
    'batch_size': 1,
    'num_workers': 1,
    'cache_strategy': 'testOnly',
    'different_resolution_strategy': 'none',
    'normalize_data': 'mean-std',
    'normalize_const': 'None',
    'downsample_factor': 1,
    'max_channels': 2,
}

data_module = MultiDataModule(**params_data)
data_module.setup(stage='test')
test_loader = data_module.test_dataloader()
data = next(iter(test_loader))

prediction, reference = strategy.predict(data, device=device, num_frames=8)
print('Done')

In [ ]:
#EDIT: Changed diffusivity B from 5e-06 to 5e-04

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import matplotlib as mpl
import seaborn as sns
import numpy as np

sns.set_theme(style="white")
cmap = sns.color_palette("twilight", as_cmap=True)
mpl.rcParams['animation.embed_limit'] = 100.0

pred_plot = prediction[0, :, 0, :, :]
ref_plot = reference[0, :, 0, :, :]

vmin = min(pred_plot.min(), ref_plot.min())
vmax = max(pred_plot.max(), ref_plot.max())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
im1 = ax1.imshow(pred_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)
im2 = ax2.imshow(ref_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)

ax1.set_title("Prediction (T=0)")
ax2.set_title("Reference (T=0)")
ax1.axis('off')
ax2.axis('off')

def update(i):
    im1.set_data(pred_plot[i])
    im2.set_data(ref_plot[i])

    ax1.set_title(f"Prediction (T={i})")
    ax2.set_title(f"Reference (T={i})")

    return [im1, im2]

ani = animation.FuncAnimation(
    fig,
    update,
    frames=8,
    interval=100, 
    blit=False
)

plt.close() 

display(HTML(ani.to_jshtml()))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import matplotlib as mpl
import seaborn as sns
import numpy as np

sns.set_theme(style="white")
cmap = sns.color_palette("twilight", as_cmap=True)
mpl.rcParams['animation.embed_limit'] = 100.0

pred_plot = prediction[0, :, 1, :, :]
ref_plot = reference[0, :, 1, :, :]

vmin = min(pred_plot.min(), ref_plot.min())
vmax = max(pred_plot.max(), ref_plot.max())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
im1 = ax1.imshow(pred_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)
im2 = ax2.imshow(ref_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)

ax1.set_title("Prediction (T=0)")
ax2.set_title("Reference (T=0)")
ax1.axis('off')
ax2.axis('off')

def update(i):
    im1.set_data(pred_plot[i])
    im2.set_data(ref_plot[i])

    ax1.set_title(f"Prediction (T={i})")
    ax2.set_title(f"Reference (T={i})")

    return [im1, im2]

ani = animation.FuncAnimation(
    fig,
    update,
    frames=8,
    interval=100, 
    blit=False
)

plt.close() 

display(HTML(ani.to_jshtml()))

# Conclusion: Changing Diffusivity B has no impact on pred

# Change in Diffusivity A
2e-05 -> 2e-06
To see if any changes in Diffusivity A would affect predictions.

In [ ]:
import h5py
import numpy as np
import scipy.io
from scipy.ndimage import zoom


file_path = '/Volumes/T7/New_Data/2d_gs_rd/data/2DRD_2x3001x100x100_[dt=05].mat'
data = scipy.io.loadmat(file_path)
uv_data = data['uv']  # (2,3001,100,100)

# (2,3001,100,100)-> (8,2,100,100)
step = 400
uv_data = uv_data[:, ::step, :, :]
uv_reshaped = np.transpose(uv_data, (1, 0, 2, 3))

#New Domain extent
L = 2.5
res = 256
coords = np.linspace(0, L, res, dtype=np.float32)

#Res 100x100 -> 256x256
zoom_factors = (1, 1, 256/100, 256/100)
uv_resized = zoom(uv_reshaped, zoom_factors, order=3).astype(np.float32)

print('New shape:', uv_resized.shape)

# Stats calculation
norm_max = np.max(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)
norm_mean = np.mean(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)
norm_min = np.min(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)
norm_std = np.std(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)

#Feed rate/kill rate
const_vals = np.array([0.04, 0.06], dtype=np.float64)

#HDF5 creation
output_path = '/Volumes/T7/New_Data/2d_gs_rd/data/gs_theta.hdf5'
with h5py.File(output_path, 'w') as f:

    # Stats
    f.create_dataset('norm_const_max', data=const_vals)
    f.create_dataset('norm_const_mean', data=const_vals)
    f.create_dataset('norm_const_min', data=const_vals)
    f.create_dataset('norm_const_std', data=np.array([0.0, 0.0], dtype=np.float64))

    f.create_dataset('norm_fields_sca_max', data=norm_max.astype(np.float64))
    f.create_dataset('norm_fields_sca_mean', data=norm_mean.astype(np.float64))
    f.create_dataset('norm_fields_sca_min', data=norm_min.astype(np.float64))
    f.create_dataset('norm_fields_sca_std', data=norm_std.astype(np.float64))
    f.create_dataset('norm_fields_std', data=norm_std.astype(np.float32))

    # Data
    sims_group = f.create_group('sims')
    sim0 = sims_group.create_dataset('sim0', data=uv_resized)

    sim0.attrs['Feed Rate'] = 0.04
    sim0.attrs['Kill Rate'] = 0.06

    # Metadata
    attrs = {
        'PDE': 'Gray Scott',
        'Dimension': 2,
        'Fields Scheme': 'cb',
        'Fields': ['Concentration A', 'Concentration B'],
        'Domain Extent': 2.5,
        'Resolution': 256,
        'Time Steps': 8,
        'Dt': 200.0,
        'Sub Steps': 0,
        'Constants': ['Feed Rate', 'Kill Rate'],
        'Initial Types': ['Gaussian Blobs'],
        'Diffusivity A': 2e-06,
        'Diffusivity B': 5e-06,
        'Dynamics Types (Feed Rate, Kill Rate)': [[0.04, 0.06]], #change to check prediction
        'Boundary Conditions Order': ['x negative', 'x positive', 'y negative', 'y positive'],
        'Boundary Conditions': ['periodic', 'periodic', 'periodic', 'periodic']
    }

    for key, val in attrs.items():
        sims_group.attrs[key] = val

print('Done')
import matplotlib.pyplot as plt
# Compare a slice of original vs resized
plt.subplot(1,2,1); plt.imshow(uv_reshaped[0,0]); plt.title("Original (100x100)")
plt.subplot(1,2,2); plt.imshow(uv_resized[0,0]); plt.title("Zoomed (256x256)")
plt.show()

In [ ]:
from pdetransformer.data import MultiDataModule

dataset = 'gs_theta'
dataset_name = 'gs_theta'

params_data = {
    'path_index':
        {'2D_APE_xxl': '/Volumes/T7/New_Data/2d_gs_rd/data'},
    'dataset_names': [ dataset ],
    'dataset_type': '2D_APE_xxl',
    'unrolling_steps': 1,
    'test_unrolling_steps': 7,
    'batch_size': 1,
    'num_workers': 1,
    'cache_strategy': 'testOnly',
    'different_resolution_strategy': 'none',
    'normalize_data': 'mean-std',
    'normalize_const': 'None',
    'downsample_factor': 1,
    'max_channels': 2,
}

data_module = MultiDataModule(**params_data)
data_module.setup(stage='test')
test_loader = data_module.test_dataloader()
data = next(iter(test_loader))

prediction, reference = strategy.predict(data, device=device, num_frames=8)
print('Done')

In [ ]:
#EDIT: Changed diffusivity A from 2e-05 to 2e-06

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import matplotlib as mpl
import seaborn as sns
import numpy as np

sns.set_theme(style="white")
cmap = sns.color_palette("twilight", as_cmap=True)
mpl.rcParams['animation.embed_limit'] = 100.0

pred_plot = prediction[0, :, 0, :, :]
ref_plot = reference[0, :, 0, :, :]

vmin = min(pred_plot.min(), ref_plot.min())
vmax = max(pred_plot.max(), ref_plot.max())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
im1 = ax1.imshow(pred_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)
im2 = ax2.imshow(ref_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)

ax1.set_title("Prediction (T=0)")
ax2.set_title("Reference (T=0)")
ax1.axis('off')
ax2.axis('off')

def update(i):
    im1.set_data(pred_plot[i])
    im2.set_data(ref_plot[i])

    ax1.set_title(f"Prediction (T={i})")
    ax2.set_title(f"Reference (T={i})")

    return [im1, im2]

ani = animation.FuncAnimation(
    fig,
    update,
    frames=8,
    interval=100, 
    blit=False
)

plt.close() 

display(HTML(ani.to_jshtml()))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import matplotlib as mpl
import seaborn as sns
import numpy as np

sns.set_theme(style="white")
cmap = sns.color_palette("twilight", as_cmap=True)
mpl.rcParams['animation.embed_limit'] = 100.0

pred_plot = prediction[0, :, 1, :, :]
ref_plot = reference[0, :, 1, :, :]

vmin = min(pred_plot.min(), ref_plot.min())
vmax = max(pred_plot.max(), ref_plot.max())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
im1 = ax1.imshow(pred_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)
im2 = ax2.imshow(ref_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)

ax1.set_title("Prediction (T=0)")
ax2.set_title("Reference (T=0)")
ax1.axis('off')
ax2.axis('off')

def update(i):
    im1.set_data(pred_plot[i])
    im2.set_data(ref_plot[i])

    ax1.set_title(f"Prediction (T={i})")
    ax2.set_title(f"Reference (T={i})")

    return [im1, im2]

ani = animation.FuncAnimation(
    fig,
    update,
    frames=8,
    interval=100, 
    blit=False
)

plt.close() 

display(HTML(ani.to_jshtml()))

# Conclusion: Changing Diffusivity A has no impact on pred

# Changing feed/kill rate
[0.04,0.06] -> [0.028,0.056]
To see if the change in constants would affect predictions.

In [ ]:
import h5py
import numpy as np
import scipy.io
from scipy.ndimage import zoom


file_path = '/Volumes/T7/New_Data/2d_gs_rd/data/2DRD_2x3001x100x100_[dt=05].mat'
data = scipy.io.loadmat(file_path)
uv_data = data['uv']  # (2,3001,100,100)

# (2,3001,100,100)-> (8,2,100,100)
step = 400
uv_data = uv_data[:, ::step, :, :]
uv_reshaped = np.transpose(uv_data, (1, 0, 2, 3))

#New Domain extent
L = 2.5
res = 256
coords = np.linspace(0, L, res, dtype=np.float32)

#Res 100x100 -> 256x256
zoom_factors = (1, 1, 256/100, 256/100)
uv_resized = zoom(uv_reshaped, zoom_factors, order=3).astype(np.float32)

print('New shape:', uv_resized.shape)

# Stats calculation
norm_max = np.max(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)
norm_mean = np.mean(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)
norm_min = np.min(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)
norm_std = np.std(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)

#Feed rate/kill rate
const_vals = np.array([0.028, 0.056], dtype=np.float64)

#HDF5 creation
output_path = '/Volumes/T7/New_Data/2d_gs_rd/data/gs_theta.hdf5'
with h5py.File(output_path, 'w') as f:

    # Stats
    f.create_dataset('norm_const_max', data=const_vals)
    f.create_dataset('norm_const_mean', data=const_vals)
    f.create_dataset('norm_const_min', data=const_vals)
    f.create_dataset('norm_const_std', data=np.array([0.0, 0.0], dtype=np.float64))

    f.create_dataset('norm_fields_sca_max', data=norm_max.astype(np.float64))
    f.create_dataset('norm_fields_sca_mean', data=norm_mean.astype(np.float64))
    f.create_dataset('norm_fields_sca_min', data=norm_min.astype(np.float64))
    f.create_dataset('norm_fields_sca_std', data=norm_std.astype(np.float64))
    f.create_dataset('norm_fields_std', data=norm_std.astype(np.float32))

    # Data
    sims_group = f.create_group('sims')
    sim0 = sims_group.create_dataset('sim0', data=uv_resized)

    sim0.attrs['Feed Rate'] = 0.028
    sim0.attrs['Kill Rate'] = 0.056

    # Metadata
    attrs = {
        'PDE': 'Gray Scott',
        'Dimension': 2,
        'Fields Scheme': 'cb',
        'Fields': ['Concentration A', 'Concentration B'],
        'Domain Extent': 2.5,
        'Resolution': 256,
        'Time Steps': 8,
        'Dt': 200.0,
        'Sub Steps': 0,
        'Constants': ['Feed Rate', 'Kill Rate'],
        'Initial Types': ['Gaussian Blobs'],
        'Diffusivity A': 2e-04,
        'Diffusivity B': 5e-06,
        'Dynamics Types (Feed Rate, Kill Rate)': [[0.028, 0.056]],
        'Boundary Conditions Order': ['x negative', 'x positive', 'y negative', 'y positive'],
        'Boundary Conditions': ['periodic', 'periodic', 'periodic', 'periodic']
    }

    for key, val in attrs.items():
        sims_group.attrs[key] = val

print('Done')
import matplotlib.pyplot as plt
# Compare a slice of original vs resized
plt.subplot(1,2,1); plt.imshow(uv_reshaped[0,0]); plt.title("Original (100x100)")
plt.subplot(1,2,2); plt.imshow(uv_resized[0,0]); plt.title("Zoomed (256x256)")
plt.show()

In [ ]:
from pdetransformer.data import MultiDataModule

dataset = 'gs_theta'
dataset_name = 'gs_theta'

params_data = {
    'path_index':
        {'2D_APE_xxl': '/Volumes/T7/New_Data/2d_gs_rd/data'},
    'dataset_names': [ dataset ],
    'dataset_type': '2D_APE_xxl',
    'unrolling_steps': 1,
    'test_unrolling_steps': 7,
    'batch_size': 1,
    'num_workers': 1,
    'cache_strategy': 'testOnly',
    'different_resolution_strategy': 'none',
    'normalize_data': 'mean-std',
    'normalize_const': 'None',
    'downsample_factor': 1,
    'max_channels': 2,
}

data_module = MultiDataModule(**params_data)
data_module.setup(stage='test')
test_loader = data_module.test_dataloader()
data = next(iter(test_loader))

prediction1, reference1 = strategy.predict(data, device=device, num_frames=8)
print('Done')

In [ ]:
#Changed feed/kill rate to 0.028/0.056 respectively

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import matplotlib as mpl
import seaborn as sns
import numpy as np

sns.set_theme(style="white")
cmap = sns.color_palette("twilight", as_cmap=True)
mpl.rcParams['animation.embed_limit'] = 100.0

pred_plot = prediction[0, :, 0, :, :]
ref_plot = reference[0, :, 0, :, :]

vmin = min(pred_plot.min(), ref_plot.min())
vmax = max(pred_plot.max(), ref_plot.max())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
im1 = ax1.imshow(pred_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)
im2 = ax2.imshow(ref_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)

ax1.set_title("Prediction (T=0)")
ax2.set_title("Reference (T=0)")
ax1.axis('off')
ax2.axis('off')

def update(i):
    im1.set_data(pred_plot[i])
    im2.set_data(ref_plot[i])

    ax1.set_title(f"Prediction (T={i})")
    ax2.set_title(f"Reference (T={i})")

    return [im1, im2]

ani = animation.FuncAnimation(
    fig,
    update,
    frames=8,
    interval=100, 
    blit=False
)

plt.close()

display(HTML(ani.to_jshtml()))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import matplotlib as mpl
import seaborn as sns
import numpy as np

sns.set_theme(style="white")
cmap = sns.color_palette("twilight", as_cmap=True)
mpl.rcParams['animation.embed_limit'] = 100.0

pred_plot = prediction[0, :, 1, :, :]
ref_plot = reference[0, :, 1, :, :]

vmin = min(pred_plot.min(), ref_plot.min())
vmax = max(pred_plot.max(), ref_plot.max())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
im1 = ax1.imshow(pred_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)
im2 = ax2.imshow(ref_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)

ax1.set_title("Prediction (T=0)")
ax2.set_title("Reference (T=0)")
ax1.axis('off')
ax2.axis('off')

def update(i):
    im1.set_data(pred_plot[i])
    im2.set_data(ref_plot[i])

    ax1.set_title(f"Prediction (T={i})")
    ax2.set_title(f"Reference (T={i})")

    return [im1, im2]

ani = animation.FuncAnimation(
    fig,
    update,
    frames=8,
    interval=100,
    blit=False
)

plt.close() 

display(HTML(ani.to_jshtml()))

In [ ]:
import h5py

file = '/Volumes/T7/New_Data/2d_gs_rd/data/gs_theta.hdf5'

with h5py.File(file, 'r') as f:
    print(f.keys())
    print(f['norm_const_max'])  # [0.04,0.06]
    print(f['norm_const_mean']) # [0.04,0.06]
    print(f['norm_const_min']) # [0.04,0.06]
    print(f['norm_const_std']) # [0,0] floating point

    print()
    print(f['norm_fields_sca_max']) # [[[1.00000179]], [[0.41890338]]]
    print(f['norm_fields_sca_mean']) # [[[0.69602691]], [[0.12202653]]]
    print(f['norm_fields_sca_min']) # [[[ 2.51750410e-01]], [[-4.32133675e-07]]]
    print(f['norm_fields_sca_std']) # [[[0.25863836]], [[0.12579946]]]

    print()
    print(f['norm_fields_std']) # [[[0.25863832]], [[0.12579946]]]

    print()
    print(f['sims'])
    print(f['sims'].keys())
    print(f['sims']['sim0'])

    print()
    for attrsKeys in f['sims'].attrs.keys():
        print(f'{attrsKeys}:',f['sims'].attrs[attrsKeys])

    print()
    print(f['sims']['sim0'].attrs.keys())
    print(f['sims']['sim0'].attrs['Feed Rate'])
    print(f['sims']['sim0'].attrs['Kill Rate'])
    print(f['norm_const_max'][:])
    print(f['norm_const_mean'][:])
    print(f['norm_const_min'][:])
    print(f['norm_const_std'][:])


In [ ]:
#constants are zeroed out in line 1499 of pde_transformer.py, self.PDE_PARAMETER_SCALING == 0 in init
#Ultimately, constants does not seem to affect pred

# More changes in feed/kill rate
This time, both feed/kill rate were zeroed out for testing.

In [ ]:
import h5py
import numpy as np
import scipy.io
from scipy.ndimage import zoom


file_path = '/Volumes/T7/New_Data/2d_gs_rd/data/2DRD_2x3001x100x100_[dt=05].mat'
data = scipy.io.loadmat(file_path)
uv_data = data['uv']  # (2,3001,100,100)

# (2,3001,100,100)-> (8,2,100,100)
step = 400
uv_data = uv_data[:, ::step, :, :]
uv_reshaped = np.transpose(uv_data, (1, 0, 2, 3))

#New Domain extent
L = 2.5
res = 256
coords = np.linspace(0, L, res, dtype=np.float32)

#Res 100x100 -> 256x256
zoom_factors = (1, 1, 256/100, 256/100)
uv_resized = zoom(uv_reshaped, zoom_factors, order=3).astype(np.float32)

print('New shape:', uv_resized.shape)

# Stats calculation
norm_max = np.max(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)
norm_mean = np.mean(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)
norm_min = np.min(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)
norm_std = np.std(uv_resized, axis=(0, 2, 3)).reshape(2, 1, 1)

#Feed rate/kill rate
const_vals = np.array([0.0, 0.0], dtype=np.float64)

#HDF5 creation
output_path = '/Volumes/T7/New_Data/2d_gs_rd/data/gs_theta2.hdf5'
with h5py.File(output_path, 'w') as f:

    # Stats
    f.create_dataset('norm_const_max', data=const_vals)
    f.create_dataset('norm_const_mean', data=const_vals)
    f.create_dataset('norm_const_min', data=const_vals)
    f.create_dataset('norm_const_std', data=np.array([0.0, 0.0], dtype=np.float64))

    f.create_dataset('norm_fields_sca_max', data=norm_max.astype(np.float64))
    f.create_dataset('norm_fields_sca_mean', data=norm_mean.astype(np.float64))
    f.create_dataset('norm_fields_sca_min', data=norm_min.astype(np.float64))
    f.create_dataset('norm_fields_sca_std', data=norm_std.astype(np.float64))
    f.create_dataset('norm_fields_std', data=norm_std.astype(np.float32))

    # Data
    sims_group = f.create_group('sims')
    sim0 = sims_group.create_dataset('sim0', data=uv_resized)

    sim0.attrs['Feed Rate'] = 0.0
    sim0.attrs['Kill Rate'] = 0.0

    # Metadata
    attrs = {
        'PDE': 'Gray Scott',
        'Dimension': 2,
        'Fields Scheme': 'cb',
        'Fields': ['Concentration A', 'Concentration B'],
        'Domain Extent': 2.5,
        'Resolution': 256,
        'Time Steps': 8,
        'Dt': 200.0,
        'Sub Steps': 0,
        'Constants': ['Feed Rate', 'Kill Rate'],
        'Initial Types': ['Gaussian Blobs'],
        'Diffusivity A': 2e-04,
        'Diffusivity B': 5e-06,
        'Dynamics Types (Feed Rate, Kill Rate)': [[0.0, 0.0]],
        'Boundary Conditions Order': ['x negative', 'x positive', 'y negative', 'y positive'],
        'Boundary Conditions': ['periodic', 'periodic', 'periodic', 'periodic']
    }

    for key, val in attrs.items():
        sims_group.attrs[key] = val

print('Done')
import matplotlib.pyplot as plt
# Compare a slice of original vs resized
plt.subplot(1,2,1); plt.imshow(uv_reshaped[0,0]); plt.title("Original (100x100)")
plt.subplot(1,2,2); plt.imshow(uv_resized[0,0]); plt.title("Zoomed (256x256)")
plt.show()

In [ ]:
import h5py

file = '/Volumes/T7/New_Data/2d_gs_rd/data/gs_theta2.hdf5'

with h5py.File(file, 'r') as f:
    print(f.keys())
    print(f['norm_const_max'])  # [0.04,0.06]
    print(f['norm_const_mean']) # [0.04,0.06]
    print(f['norm_const_min']) # [0.04,0.06]
    print(f['norm_const_std']) # [0,0] floating point

    print()
    print(f['norm_fields_sca_max']) # [[[1.00000179]], [[0.41890338]]]
    print(f['norm_fields_sca_mean']) # [[[0.69602691]], [[0.12202653]]]
    print(f['norm_fields_sca_min']) # [[[ 2.51750410e-01]], [[-4.32133675e-07]]]
    print(f['norm_fields_sca_std']) # [[[0.25863836]], [[0.12579946]]]

    print()
    print(f['norm_fields_std']) # [[[0.25863832]], [[0.12579946]]]

    print()
    print(f['sims'])
    print(f['sims'].keys())
    print(f['sims']['sim0'])

    print()
    for attrsKeys in f['sims'].attrs.keys():
        print(f'{attrsKeys}:',f['sims'].attrs[attrsKeys])

    print()
    print(f['sims']['sim0'].attrs.keys())
    print(f['sims']['sim0'].attrs['Feed Rate'])
    print(f['sims']['sim0'].attrs['Kill Rate'])
    print(f['norm_const_max'][:])
    print(f['norm_const_mean'][:])
    print(f['norm_const_min'][:])
    print(f['norm_const_std'][:])


In [ ]:
from pdetransformer.data import MultiDataModule

dataset = 'gs_theta'
dataset_name = 'gs_theta'

params_data = {
    'path_index':
        {'2D_APE_xxl': '/Volumes/T7/New_Data/2d_gs_rd'},
    'dataset_names': [ dataset ],
    'dataset_type': '2D_APE_xxl',
    'unrolling_steps': 1,
    'test_unrolling_steps': 7,
    'batch_size': 1,
    'num_workers': 1,
    'cache_strategy': 'testOnly',
    'different_resolution_strategy': 'none',
    'normalize_data': 'mean-std',
    'normalize_const': 'None',
    'downsample_factor': 1,
    'max_channels': 2,
}

data_module = MultiDataModule(**params_data)
data_module.setup(stage='test')
test_loader = data_module.test_dataloader()
data = next(iter(test_loader))

prediction2, reference2 = strategy.predict(data, device=device, num_frames=8)
print('Done')

In [ ]:
#Changed feed/kill rate to 0.0 for both

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import matplotlib as mpl
import seaborn as sns
import numpy as np

sns.set_theme(style="white")
cmap = sns.color_palette("twilight", as_cmap=True)
mpl.rcParams['animation.embed_limit'] = 100.0

pred_plot = prediction[0, :, 0, :, :]
ref_plot = reference[0, :, 0, :, :]

vmin = min(pred_plot.min(), ref_plot.min())
vmax = max(pred_plot.max(), ref_plot.max())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
im1 = ax1.imshow(pred_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)
im2 = ax2.imshow(ref_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)

ax1.set_title("Prediction (T=0)")
ax2.set_title("Reference (T=0)")
ax1.axis('off')
ax2.axis('off')

def update(i):
    im1.set_data(pred_plot[i])
    im2.set_data(ref_plot[i])

    ax1.set_title(f"Prediction (T={i})")
    ax2.set_title(f"Reference (T={i})")

    return [im1, im2]

ani = animation.FuncAnimation(
    fig,
    update,
    frames=8,
    interval=100, 
    blit=False
)

plt.close() 


display(HTML(ani.to_jshtml()))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import matplotlib as mpl
import seaborn as sns
import numpy as np

sns.set_theme(style="white")
cmap = sns.color_palette("twilight", as_cmap=True)
mpl.rcParams['animation.embed_limit'] = 100.0

pred_plot = prediction[0, :, 1, :, :]
ref_plot = reference[0, :, 1, :, :]

vmin = min(pred_plot.min(), ref_plot.min())
vmax = max(pred_plot.max(), ref_plot.max())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
im1 = ax1.imshow(pred_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)
im2 = ax2.imshow(ref_plot[0], cmap=cmap, vmin=vmin, vmax=vmax)

ax1.set_title("Prediction (T=0)")
ax2.set_title("Reference (T=0)")
ax1.axis('off')
ax2.axis('off')

def update(i):
    im1.set_data(pred_plot[i])
    im2.set_data(ref_plot[i])

    ax1.set_title(f"Prediction (T={i})")
    ax2.set_title(f"Reference (T={i})")

    return [im1, im2]

ani = animation.FuncAnimation(
    fig,
    update,
    frames=8,
    interval=100, 
    blit=False
)

plt.close() 

display(HTML(ani.to_jshtml()))

In [ ]:
from pdetransformer.metric.mse import SimulationMSE
import torch

nrmse_metric = SimulationMSE(
    frequency=1,
    num_tasks=1,
    max_length=3001,
    normalized=True,
    dataset_names=['gs_theta']
)

pred_tensor = torch.from_numpy(prediction)
ref_tensor = torch.from_numpy(reference)

class_labels = torch.tensor([0])
nrmse_metric.update(pred_tensor, ref_tensor, class_labels)

nrmse_results = nrmse_metric.compute()

print("Times tracked by metric:", nrmse_metric.times)

for i in range(len(nrmse_results[0])):
    print(f"Index {i}, timestep {nrmse_metric.times[i]}: {nrmse_results[0,i].item():.6f}")

times = nrmse_metric.times

step1_index = times.index(1)
step3000_index = times.index(3000)

nrmse_step_1 = nrmse_results[0, step1_index].item()
nrmse_step_3000 = nrmse_results[0, step3000_index].item()

print(f"NRMSE after 1 step:  {nrmse_step_1:.6f}")
print(f"NRMSE after 3000 steps: {nrmse_step_3000:.6f}")

# Conclusion: Any changes in feed/kill rate does not affect predictions at all